# Recuperación multi-salto sobre HotpotQA: el índice, el grafo y el proceso de consulta, paso a paso

Este cuaderno recorre el sistema evaluado el 30 de agosto de 2026 (**versión v4**: memoria canónica curada,
enlazador de nombre a nombre y Plan A generalizado) sobre el **índice real del banco de HotpotQA**
(1 000 preguntas, corpus de 9 811 pasajes; medición única: R@5 94,7 / FC@5 90,2 con `gpt-4o-mini`).
Es el complemento de `metodologia_recuperacion.ipynb` (2Wiki, v3): allí se traza el sistema del documento;
aquí, la configuración congelada con la que se generalizó a un conjunto nuevo **sin ningún ajuste**.

| parte | qué muestra |
|---|---|
| 1. Los dos bancos | qué son 2Wiki y HotpotQA, con preguntas y pasajes oro reales |
| 2. El índice, capa a capa | extracción OpenIE → memoria canónica (diccionario) → grafo canónico → almacenes vectoriales |
| 3. Navegar el índice | de un pasaje a sus hechos y entidades; de una entidad a sus alias, su pasaje propio y su vecindario |
| 4. La consulta, etapa a etapa | enlazador (escalones 1 y 2) → afinidad → candidatos K(q) → analista (plan) → siembra → paseo → salto dirigido → conjunto final |
| 5. Dos preguntas reales | una de puente y una de comparación, con cada producto intermedio y las métricas |
| 6. Contrafactuales | sin salto, sin conjunto, sin modulación: qué cambia en la misma pregunta |
| 7. Un fallo real | dónde queda el oro cuando el sistema falla |
| 8. El banco medido | agregados por tipo, referencias publicadas con los mismos modelos, pregunta libre y lote opcional |

**Cómo ejecutarlo.** Desde la raíz del repositorio, con el entorno `venv` y `.env` (clave de OpenAI). La carga del grafo
(146 274 nodos) tarda 1–2 minutos. Las llamadas al modelo de las preguntas del banco están en la caché SQLite
(`artifacts/wiki2/plan_cache.sqlite`), así que re-ejecutar no factura; la pregunta libre sí (céntimos).
Todas las funciones de traza reproducen lo que hace `Wiki2PlanRAG.buscar()` (`src/asistente_vih/retrieval/wiki2_plan.py`)
y se comprueban contra él al final.

In [1]:
import os, sys, json, re, collections
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():          # permite ejecutar desde notebooks/ o desde la raíz
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)
W = ROOT / "artifacts" / "wiki2"

from asistente_vih.eval.wiki2 import cargar_split, titulos_oro, DATASETS, BM25Wiki
from asistente_vih.retrieval.wiki2_catrag import spans_entidad, _plegar_alias, _plegar_con_desamb, clasificar_tipo
from asistente_vih.retrieval.wiki2_plan import Wiki2PlanRAG, _PROMPT_ANALISTA, _PROMPT_SALTO, _PROMPT_CONJUNTO

preguntas, corpus = cargar_split("hotpot_benchmark")
textos = {c["title"]: c["text"] for c in corpus}
print(f"HotpotQA (subset de reproducción de HippoRAG 2 / CatRAG): {len(preguntas)} preguntas · corpus {len(corpus)} pasajes")
print("tipos:", dict(collections.Counter(q["type"] for q in preguntas)), "| nivel:", dict(collections.Counter(q["level"] for q in preguntas)))

HotpotQA (subset de reproducción de HippoRAG 2 / CatRAG): 1000 preguntas · corpus 9811 pasajes
tipos: {'bridge': 811, 'comparison': 189} | nivel: {'hard': 1000}


## 1. Los dos bancos de pruebas

Un **banco** es, para nosotros, un conjunto de preguntas con sus pasajes oro más un corpus cerrado sobre el que se recupera
*abiertamente* (contra todos los pasajes, no contra los 10 que trae cada pregunta). Usamos los subconjuntos de
reproducción publicados con HippoRAG 2 —los mismos que usa CatRAG—, de modo que las cifras publicadas con los mismos
modelos (`gpt-4o-mini` + `text-embedding-3-small`) son la referencia directa.

| | **2WikiMultihopQA** | **HotpotQA** |
|---|---|---|
| origen | 2020; preguntas generadas por **plantillas** desde tripletes de Wikidata sobre introducciones de Wikipedia | 2018; preguntas **escritas por personas** sobre pares de introducciones de Wikipedia |
| banco | 1 000 preguntas del dev; corpus = unión de sus contextos: 6 119 pasajes | 1 000 preguntas del dev (todas de nivel *hard*); corpus 9 811 pasajes |
| tipos | comparación (244), composicional (413), inferencia (108), comparación-puente (235) | puente (811), comparación (189) |
| pasajes oro | 2, salvo comparación-puente: 4 | siempre 2 |
| oro extra | tripletes de evidencia con **QIDs de Wikidata** (auditoría del diccionario) | frases de apoyo (`supporting_facts` a nivel de frase); sin identificadores externos |
| estilo | corto y regular («Who is the mother of the director of film X?»), entidades en Title Case | más largo y libre (16 palabras de media frente a 12), menciones descriptivas, minúsculas |
| sondeo (retenido, para decidir) | 200 = 50 por tipo, del dev restante; mini-corpus 1 558 | 200 = 100 por tipo, del train (nivel *hard*); mini-corpus 1 990 |

Lo que se mide es lo mismo en los dos: `recall@k` (fracción de títulos oro en el top-k) y `full_chain@k`
(**todos** los oros en el top-k, la métrica que exige la cadena completa). Dos ejemplos reales de HotpotQA:

In [2]:
def mostrar(q, n=170):
    oro = list(dict.fromkeys(t for t, _ in q["supporting_facts"]))
    print(f"[{q['type']}] {q['question']}\n   respuesta: {q['answer']} | pasajes oro: {oro}")
    for t in oro:
        print(f"   «{t}»: {textos[t][:n]}…")

por_id = {q["_id"]: q for q in preguntas}
def buscar_pregunta(inicio): return next(q for q in preguntas if q["question"].startswith(inicio))
q_puente = buscar_pregunta("What 2003 Christmas-themed romantic comedy did Gregor Fisher")
q_comp = buscar_pregunta("Which rock band chose its name by drawing it out of a hat")
mostrar(q_puente); print(); mostrar(q_comp)

[bridge] What 2003 Christmas-themed romantic comedy did Gregor Fisher have a role in?
   respuesta: Love Actually | pasajes oro: ['Gregor Fisher', 'Love Actually']
   «Gregor Fisher»: Gregor Fisher (born 22 December 1953) is a Scottish comedian and actor. He is perhaps best known for his long portrayal as protagonist and suffering Glasgow alcoholic Rab…
   «Love Actually»: Love Actually is a 2003 Christmas-themed romantic comedy film written and directed by Richard Curtis. It features an ensemble cast, many of whom had worked with Curtis in…

[comparison] Which rock band chose its name by drawing it out of a hat, Switchfoot or Midnight Oil?
   respuesta: Midnight Oil | pasajes oro: ['Midnight Oil', 'Switchfoot']
   «Midnight Oil»: Midnight Oil (also known informally as "The Oils" to fans) are an Australian rock band, who originally performed as Farm from 1972 with drummer Rob Hirst, bass guitarist …
   «Switchfoot»: Switchfoot is an American alternative rock band from San Diego, Califo

In [3]:
# y, para contraste, la forma regular de 2Wiki (mismo esquema de campos: _id, question, type, supporting_facts)
p2, c2 = cargar_split("benchmark")
t2 = {c["title"]: c["text"] for c in c2}
for tipo in ("compositional", "comparison", "inference", "bridge_comparison"):
    q = next(x for x in p2 if x["type"] == tipo)
    print(f"[{tipo:18s}] {q['question']}   oro: {sorted(titulos_oro(q))}")
print(f"\nlongitud media (palabras): 2Wiki {np.mean([len(q['question'].split()) for q in p2]):.1f} · HotpotQA {np.mean([len(q['question'].split()) for q in preguntas]):.1f}")
print("preguntas de HotpotQA sin ninguna palabra en mayúscula tras la primera:",
      sum(1 for q in preguntas if not re.search(r"\b[A-Z][a-z]", q["question"][1:])), "(en 2Wiki: 1)")

[compositional     ] When did Lothair Ii's mother die?   oro: ['Ermengarde of Tours', 'Lothair II']
[comparison        ] Which film was released first, Aas Ka Panchhi or Phoolwari?   oro: ['Aas Ka Panchhi', 'Phoolwari']
[inference         ] Who is Raghnall Mac Ruaidhrí's paternal grandfather?   oro: ['Raghnall Mac Ruaidhrí', 'Ruaidhrí Mac Ruaidhrí']
[bridge_comparison ] Which film has the director who is older, God'S Gift To Women or Aldri Annet Enn Bråk?   oro: ['Aldri annet enn bråk', 'Edith Carlmar', "God's Gift to Women", 'Michael Curtiz']

longitud media (palabras): 2Wiki 12.0 · HotpotQA 16.0
preguntas de HotpotQA sin ninguna palabra en mayúscula tras la primera: 35 (en 2Wiki: 1)


## 2. El índice, capa a capa (§2.3 del documento)

La indexación es la misma cadena que en 2Wiki, sin cambios: **(a)** extracción OpenIE reificada por pasaje
(`ingest/wiki2_openie.py`, `gpt-4o-mini`), **(b)** memoria canónica de entidades (`ingest/wiki2_entidades.py`, v4),
**(c)** grafo canónico (`ingest/wiki2_grafo.py`) y **(d)** almacenes vectoriales alineados por identificador de nodo.
Vamos a mirar cada capa con un pasaje real: **«Love Actually»**, el puente de la primera pregunta.

In [4]:
openie = {}
with open(W / "openie_hotpot_benchmark.jsonl", encoding="utf-8") as f:
    for l in f:
        d = json.loads(l); openie[d["chunk_id"]] = d
n_aser = sum(len(d["aserciones"]) for d in openie.values()); n_men = sum(len(d["entidades"]) for d in openie.values())
print(f"(a) OpenIE: {len(openie)} pasajes → {n_aser} aserciones reificadas y {n_men} menciones de entidad "
      f"({n_aser/len(openie):.1f} y {n_men/len(openie):.1f} por pasaje)")
d = openie["Love Actually"]
print(f"\nPasaje «Love Actually» — entidades extraídas ({len(d['entidades'])}): {d['entidades']}")
print("aserciones (sujeto | relación | objeto | frase autocontenida):")
for a in d["aserciones"][:8]:
    print(f"   {a['id_asercion'][-8:]}  {a['sujeto']} | {a['relacion_base']} | {a['objeto'][:40]}  →  «{a['descripcion_relacion'][:95]}»")

(a) OpenIE: 9811 pasajes → 87629 aserciones reificadas y 74947 menciones de entidad (8.9 y 7.6 por pasaje)

Pasaje «Love Actually» — entidades extraídas (3): ['Love Actually', 'Richard Curtis', 'London']
aserciones (sujeto | relación | objeto | frase autocontenida):
   ally::a0  Love Actually | RELEASE_DATE | 2003  →  «Love Actually is a 2003 Christmas-themed romantic comedy film.»
   ally::a1  Love Actually | DIRECTED | Richard Curtis  →  «Love Actually was written and directed by Richard Curtis.»
   ally::a2  Love Actually | BASED_ON | different aspects of love  →  «The screenplay delves into different aspects of love as shown through ten separate stories.»
   ally::a3  Love Actually | FILMED_IN | London  →  «Most of the film was filmed on location in London.»
   ally::a4  Love Actually | SET_IN | five weeks before Christmas  →  «The story begins five weeks before Christmas.»
   ally::a5  Love Actually | SET_IN | one month later  →  «The story is played out in a weekly countdown unti

**Qué es una aserción reificada.** Cada hecho del pasaje se convierte en un objeto con identidad
(`id_asercion` = `<título>::<n>`), un sujeto, una relación base, un objeto y, sobre todo, una **frase autocontenida**
(`descripcion_relacion`) con nombres completos y sin pronombres. Es esa frase la que se incrusta (vector) y la que
después lee el analista; los literales (fechas, oficios) viajan dentro de la frase y no generan nodos.

In [5]:
# (b) memoria canónica v4: una entrada por entidad real, con alias, descripción, pasajes y PROCEDENCIA de sus fusiones
entradas = {e["id"]: e for e in json.loads((W / "entidades_hotpot_benchmark_v4.json").read_text(encoding="utf-8"))}
uniones = [json.loads(l) for l in open(W / "entidades_uniones_hotpot_benchmark_v4.jsonl", encoding="utf-8")]
print(f"(b) memoria canónica: {n_men + sum(1 for d in openie.values())} fichas por mención (incluidos títulos) → {len(entradas)} entradas; "
      f"{sum(1 for e in entradas.values() if e['n_menciones'] > 1)} con más de una mención")
print("uniones registradas por origen:", dict(collections.Counter(u["origen"] for u in uniones)))
def entrada_de_titulo(t): return next((e for e in entradas.values() if e.get("pasaje_propio") == t), None)
for t in ("Love Actually", "Gregor Fisher"):
    e = entrada_de_titulo(t)
    print(f"\n{e['id']}: nombre «{e['nombre']}» · alias {e['alias']} · pasaje propio «{e['pasaje_propio']}» · aparece en {len(e['pasajes'])} pasajes · orígenes {e.get('origenes')}")
    print(f"   descripción: {e['descripcion'][:200]}…")

(b) memoria canónica: 84758 fichas por mención (incluidos títulos) → 48856 entradas; 9525 con más de una mención
uniones registradas por origen: {'auto_nombre': 102592, 'auto_casi': 658, 'titulo_sintetico': 2017, 'juez_vecinos': 6664, 'juez_titulo_sintetico_empate': 447, 'juez_titulo_mencion': 1975}

can:love actually: nombre «Love Actually» · alias ['Love Actually'] · pasaje propio «Love Actually» · aparece en 5 pasajes · orígenes {'auto_nombre': 10}
   descripción: Love Actually is a 2003 Christmas-themed romantic comedy film Love Actually was written and directed by Richard Curtis The screenplay delves into different aspects of love as shown through ten separat…

can:gregor fisher: nombre «Gregor Fisher» · alias ['Gregor Fisher'] · pasaje propio «Gregor Fisher» · aparece en 8 pasajes · orígenes {'auto_nombre': 28}
   descripción: Gregor Fisher starred in the 2006 movie Missing. Scotch and Wry included Gregor Fisher in its revolving ensemble cast. Whisky Galore! stars Gregor Fisher. 

**Qué es una entrada canónica.** Agrupa todas las formas con que el corpus nombra a una entidad (`alias`), guarda
las frases en que participa (`descripcion`, el canal de contexto), la lista de pasajes donde aparece y su
**pasaje propio** (el artículo cuyo sujeto es). En la v4 cada entrada recuerda además *quién* decidió sus fusiones
(`origenes`): nombre idéntico, casi idéntico, título→sujeto sintético o juez (vecinos, título-mención, empate).
Las variantes de superficie quedan resueltas aquí, en indexación; por eso el grafo no necesita aristas de sinonimia.

In [6]:
# Alias que colisionan (mismo alias plegado en entradas distintas): lo que el enlazador debe desambiguar por ficha
alias_multi = collections.defaultdict(set)
for e in entradas.values():
    for al in e["alias"] + [e["nombre"]]:
        pl = _plegar_alias(al)
        if pl: alias_multi[pl].add(e["id"])
colis = {a: v for a, v in alias_multi.items() if len(v) > 1}
print(f"alias plegados distintos: {len(alias_multi)} · en colisión (≥2 entradas): {len(colis)}")
ej = sorted(colis.items(), key=lambda x: -len(x[1]))[:4]
for a, v in ej:
    print(f"   «{a}» → {sorted(v)[:5]}{' …' if len(v) > 5 else ''}")

alias plegados distintos: 54342 · en colisión (≥2 entradas): 326
   «john button» → ['can:john button', 'can:john button#32171', 'can:john button#32173', 'can:john button#32180', 'can:john button#32185'] …
   «anything goes» → ['can:anything goes', 'can:anything goes#17004', 'can:anything goes#17009', 'can:anything goes#17014', 'can:anything goes#17032']
   «black book» → ['can:black book', 'can:black book#26094', 'can:black book#26098', 'can:black book#26104', 'can:black book#26115']
   «frozen» → ['can:frozen', 'can:frozen (2013 film)', 'can:frozen fever', 'can:frozen#30569']


In [7]:
# (c) y (d): cargamos el recuperador tal como se midió (esto lee el grafo canónico v4 y los vectores; 1–2 min)
r = Wiki2PlanRAG.v4(split="hotpot_benchmark", modelo="gpt-4o-mini")
tipos_nodo = collections.Counter(d.get("tipo") for _, d in r.g.nodes(data=True))
tipos_arista = collections.Counter(d.get("tipo_relacion") for _, _, d in r.g.edges(data=True))
print(f"(c) grafo canónico: {r.g.number_of_nodes()} nodos {dict(tipos_nodo)} · {r.g.number_of_edges()} aristas {dict(tipos_arista)}")
print(f"    grafo de búsqueda dirigido (roles invertidos): {len(r.nodos)} nodos · {r.A.nnz} aristas")
print(f"(d) almacenes vectoriales (text-embedding-3-small, d=1536): aserciones {r.aser_emb.shape} · fichas de entrada {r.ent_emb.shape} · pasajes {r.chunk_emb.shape}")
print(f"diales congelados (paridad HippoRAG 2): peso denso w_pas={r.peso_pasaje} · amortiguación λ={r.damping} · ancla débil ε={r.eps} · ancla del enlazador w_enl={r.peso_enlace} · modelo del analista {r.modelo}")

(c) grafo canónico: 146274 nodos {'Chunk': 9811, 'Entidad': 48834, 'Asercion': 87629} · 311979 aristas {'MENCIONADO_EN': 71887, 'EN_CHUNK': 87629, 'HAS_INTERVENTION': 84240, 'HAS_OUTCOME': 68223}
    grafo de búsqueda dirigido (roles invertidos): 146274 nodos · 311979 aristas
(d) almacenes vectoriales (text-embedding-3-small, d=1536): aserciones (87629, 1536) · fichas de entrada (48856, 1536) · pasajes (9811, 1536)
diales congelados (paridad HippoRAG 2): peso denso w_pas=0.05 · amortiguación λ=0.5 · ancla débil ε=0.05 · ancla del enlazador w_enl=0.5 · modelo del analista gpt-4o-mini


**Esquema del grafo.** Tres clases de nodo —pasaje (`chunk:<título>`), entidad canónica (`can:<nombre>`), aserción
(`<título>::<n>`)— y cuatro aristas: aserción→pasaje (procedencia, `EN_CHUNK`), aserción→entidad sujeto
(`HAS_INTERVENTION`), aserción→entidad objeto/participante (`HAS_OUTCOME`) y entidad→pasaje (`MENCIONADO_EN`).
Los nombres de las aristas de rol son los del grafo clínico (se reutiliza el mismo motor); su sentido aquí es
sujeto/objeto. Para la consulta se invierten las aristas de rol, y el resultado es un **grafo por capas**:

In [8]:
A = r.A.tocsr(); indeg = np.asarray(A.sum(0)).ravel(); outdeg = np.asarray(A.sum(1)).ravel()
es_ent = np.array([n.startswith("can:") for n in r.nodos]); es_chunk = np.array([n.startswith("chunk:") for n in r.nodos]); es_aser = ~es_ent & ~es_chunk
print(f"entidades ({es_ent.sum()}): grado de entrada máximo {indeg[es_ent].max():.0f} → nunca reciben masa del paseo: su masa es su siembra")
print(f"aserciones ({es_aser.sum()}): grado de salida mínimo {outdeg[es_aser].min():.0f} / máximo {outdeg[es_aser].max():.0f} → cada una desagua en su único pasaje")
print(f"pasajes ({es_chunk.sum()}): grado de salida máximo {outdeg[es_chunk].max():.0f} → sumideros")
print("\nConsecuencia (§2.4.6): el punto fijo del paseo tiene forma cerrada en tres canales; el paseo NO encadena saltos,")
print("los encadena la siembra (qué entidades y qué pasajes propios reciben masa). El salto dirigido existe por eso.")
print(f"\ngrado de salida de las entidades (nº de hechos + pasajes donde aparecen): mediana {np.median(outdeg[es_ent]):.0f}, máximo {outdeg[es_ent].max():.0f} "
      f"({r.nodos[int(np.argmax(np.where(es_ent, outdeg, 0)))]})")

entidades (48811): grado de entrada máximo 0 → nunca reciben masa del paseo: su masa es su siembra
aserciones (87652): grado de salida mínimo 1 / máximo 13 → cada una desagua en su único pasaje
pasajes (9811): grado de salida máximo 0 → sumideros

Consecuencia (§2.4.6): el punto fijo del paseo tiene forma cerrada en tres canales; el paseo NO encadena saltos,
los encadena la siembra (qué entidades y qué pasajes propios reciben masa). El salto dirigido existe por eso.

grado de salida de las entidades (nº de hechos + pasajes donde aparecen): mediana 2, máximo 1538 (can:history of united states prison systems)


## 3. Navegar el índice

De un pasaje a sus hechos y entidades; de una entidad a su vecindario en el grafo de búsqueda.

In [9]:
def vecindario(eid, n=8):
    e = r.entradas[eid]
    print(f"{eid} — «{e['nombre']}» · alias {e['alias'][:5]} · pasaje propio «{e['pasaje_propio']}» · en {len(e['pasajes'])} pasajes")
    salidas = [(v, d.get("tipo_relacion")) for _, v, d in r.g.in_edges(eid, data=True)]   # aristas de rol (aserción→entidad), invertidas en búsqueda
    menc = [v for _, v, d in r.g.out_edges(eid, data=True) if d.get("tipo_relacion") == "MENCIONADO_EN"]
    print(f"   hechos en los que participa: {len(salidas)} (sujeto {sum(1 for _, t in salidas if t == 'HAS_INTERVENTION')}, objeto/participante {sum(1 for _, t in salidas if t == 'HAS_OUTCOME')}) · pasajes que la mencionan: {len(menc)}")
    for a, t in salidas[:n]:
        print(f"      [{'S' if t == 'HAS_INTERVENTION' else 'O'}] {a.rsplit('::', 1)[0][:30]:30s} «{r.g.nodes[a].get('descripcion', '')[:90]}»")
    print(f"   pasajes: {[m[6:] for m in menc][:n]}")

vecindario(entrada_de_titulo("Gregor Fisher")["id"])
print(); vecindario(entrada_de_titulo("Love Actually")["id"])

can:gregor fisher — «Gregor Fisher» · alias ['Gregor Fisher'] · pasaje propio «Gregor Fisher» · en 8 pasajes
   hechos en los que participa: 18 (sujeto 12, objeto/participante 6) · pasajes que la mencionan: 8
      [S] can:gregor fisher              «»
      [O] can:gregor fisher              «»
      [O] can:gregor fisher              «»
      [O] can:gregor fisher              «»
      [O] can:gregor fisher              «»
      [S] can:gregor fisher              «»
      [S] can:gregor fisher              «»
      [S] can:gregor fisher              «»
   pasajes: ['Missing (Alvtegen novel)', 'Scotch and Wry', 'Whisky Galore! (2016 film)', 'The Baldy Man', 'Missing (2006 TV series)', 'Gregor Fisher', 'Rab C. Nesbitt', "Chewin' the Fat"]

can:love actually — «Love Actually» · alias ['Love Actually'] · pasaje propio «Love Actually» · en 5 pasajes
   hechos en los que participa: 11 (sujeto 7, objeto/participante 4) · pasajes que la mencionan: 5
      [S] can:love actually              «

In [10]:
# de un pasaje a todo lo que cuelga de él en el grafo de búsqueda
def pasaje(titulo, n=6):
    nodo = f"chunk:{titulo}"
    ents = [u for u, _, d in r.g.in_edges(nodo, data=True) if d.get("tipo_relacion") == "MENCIONADO_EN"]
    asers = r._aser_por_pasaje.get(titulo, [])
    print(f"«{titulo}»: {len(asers)} aserciones · {len(ents)} entidades canónicas: {ents[:n]}{' …' if len(ents) > n else ''}")
    for j in asers[:n]:
        print(f"   {r.aser_ids[j][-6:]}  ents={[str(x) for x in r.aser_ents[j]]}  «{r.g.nodes[r.aser_ids[j]].get('descripcion', '')[:80]}»")
pasaje("Love Actually")

«Love Actually»: 6 aserciones · 3 entidades canónicas: ['can:london royal', 'can:love actually', 'can:richard curtis']
   ly::a0  ents=['can:love actually']  «Love Actually is a 2003 Christmas-themed romantic comedy film.»
   ly::a1  ents=['can:love actually', 'can:richard curtis']  «Love Actually was written and directed by Richard Curtis.»
   ly::a2  ents=['can:love actually']  «The screenplay delves into different aspects of love as shown through ten separa»
   ly::a3  ents=['can:love actually', 'can:london royal']  «Most of the film was filmed on location in London.»
   ly::a4  ents=['can:love actually']  «The story begins five weeks before Christmas.»
   ly::a5  ents=['can:love actually']  «The story is played out in a weekly countdown until the holiday, followed by an »


## 4. La consulta, etapa a etapa (§2.4 del documento; Plan A generalizado, §sec:v4)

Las funciones siguientes reproducen `Wiki2PlanRAG.buscar()` e imprimen cada producto intermedio:

1. **Enlazador**: tramos de la pregunta → escalón 1 (alias exacto, con el paréntesis conservado y sin él) →
   escalón 2 de nombre a nombre (Dice de trigramas ≥ 0,80; coseno de nombre ≥ 0,85 solo si el léxico no admite a
   nadie; ficha como desempate) → $L(q)$.
2. **Afinidad**: un vector de consulta $v_0=e(q)$ y $\sigma_a=\langle v_0, e(\tau_a)\rangle$ para cada hecho.
3. **Candidatos** $\mathcal K(q)$ = Top-40 por $\sigma$ ∪ frontera de $L(q)$ (hechos incidentes en las entidades enlazadas).
4. **Analista** (1 llamada): menciones canónicas, **plan** de huecos, hechos seleccionados $S(q)$ y huecos sin cubrir con su **ancla**.
5. **Siembra**: (a) entidades de los hechos aprobados (compuerta dura), (b) presa densa sobre todos los pasajes,
   (c) anclas débiles, (d) anclas del enlazador y **pasaje propio** de toda entidad sembrada.
6. **Paseo** con transiciones moduladas ($\mu_a = 1/4 + 3/4\,\tilde\sigma_a$) → orden de pasajes.
7. **Salto dirigido** (si hay huecos): candidatos estructurales del pasaje propio del ancla + coseno del hueco → mini-filtro → re-siembra y nuevo paseo; rondas si quedan comodines $[X]$.
8. **Conjunto final** (1 llamada): del top-12 del paseo, los 5 que juntos cubren el plan (el top-2 del paseo se conserva).

In [11]:
_INV = ("HAS_INTERVENTION", "HAS_OUTCOME")
def desc(j): return r.g.nodes[r.aser_ids[j]].get("descripcion", "")
def marca(t, oro): return "ORO" if t in oro else ""

def etapa_enlazador(Q):
    r._preparar_alias()
    spans = spans_entidad(Q)
    print(f"  tramos por mayúsculas: {spans}")
    for t in spans:
        ex = r._exactos(t)
        print(f"    «{t}» → plegado «{_plegar_alias(t)}»{' / con paréntesis «' + _plegar_con_desamb(t) + '»' if '(' in t else ''} → escalón 1: {ex if ex else 'nada'}")
        if not ex:
            cand = r._dice_candidatos(_plegar_alias(t), n=3)
            print("       escalón 2, mejores por Dice de trigramas:", [(r.alias_lista[i], round(d, 2)) for i, d in cand])
    L = r._enlazar_menciones(Q); r._enlaces_actuales = L
    print(f"  L(q) = {L}")
    return L

def etapa_afinidad(Q, n=8):
    qv = r._qvecs(Q, "lite")[0]; sig = r.aser_emb @ qv
    print(f"  σ_a = <v_0, e(τ_a)> sobre {len(sig)} hechos: máx {sig.max():.3f} · media {sig.mean():.3f}")
    print(pd.DataFrame([(round(float(sig[j]), 3), desc(j)[:85], [str(e) for e in r.aser_ents[j]][:3]) for j in np.argsort(-sig)[:n]],
                       columns=["σ", "hecho τ_a", "entidades"]).to_string(index=False))
    return qv, sig

def etapa_candidatos(L, sig, puentes=()):
    orden = np.argsort(-sig); rango = {int(j): k + 1 for k, j in enumerate(orden)}; top40 = [int(j) for j in orden[:40]]
    frontera = r._frontera(L, sig); nuevos = [j for j in dict.fromkeys(frontera) if j not in set(top40)][:20]
    print(f"  K(q) = Top-40(σ) ∪ frontera(L(q)) = 40 + {len(nuevos)} = {40 + len(nuevos)} candidatos numerados para el analista")
    for txt in puentes:
        for i in [i for i in range(len(r.aser_ids)) if desc(i).startswith(txt)]:
            print(f"  hecho «{txt[:60]}»: σ = {sig[i]:.3f}, rango {rango[i]}, en Top-40: {i in top40}, en K(q): {i in top40 or i in nuevos}")
    return top40 + nuevos

def etapa_analista(Q, sig, L, extra=()):
    n0 = r.n_llamadas
    sel, huecos, enlaces, plan = r._analista(Q, sig, L, extra=list(extra))
    print(f"  llamada al analista ({r.modelo}; {'en caché' if r.n_llamadas == n0 else 'NUEVA'})")
    print(f"  plan: {plan}")
    print(f"  S(q), hechos seleccionados ({len(sel)}):")
    for j in sel: print(f"     · «{desc(j)[:90]}»  ents={[str(e) for e in r.aser_ents[j]][:3]}")
    print(f"  huecos sin cubrir: {huecos}")
    if len(enlaces) > len(L): print(f"  enlaces ampliados con las menciones canónicas del analista: {enlaces[len(L):]}")
    r._enlaces_actuales = enlaces
    return sel, huecos, enlaces, plan

def etapa_siembra(Q, qv, sig, apr, oro, top=10):
    stilde = sig.copy(); stilde[apr] = 1.0                      # σ̃: los hechos aprobados valen 1
    src = {}
    def add(n, k, w): src.setdefault(n, [0.0] * 4); src[n][k] += w
    pesos, cuenta = {}, {}
    for j in apr:                                               # (a) compuerta dura
        for e in r.aser_ents[j]: pesos[e] = pesos.get(e, 0) + float(np.clip(stilde[j], 0, 1)); cuenta[e] = cuenta.get(e, 0) + 1
    for e in pesos: add(e, 0, pesos[e] / cuenta[e])
    s = r.chunk_emb @ qv; mm = (s - s.min()) / (s.max() - s.min() + 1e-9)   # (b) presa densa
    for t, m in enumerate(mm):
        n = f"chunk:{r.titulos[t]}"
        if n in r.idx: add(n, 1, r.peso_pasaje * float(m))
    se = r.ent_emb @ qv                                          # (c) anclas débiles
    for j in np.argsort(-se)[:2]:
        n = str(r.ent_ids[int(j)])
        if n in r.idx: add(n, 2, r.eps)
    for eid in r._enlaces_actuales:                              # (d) enlazador
        tot = sum(src.get(eid, [0] * 4))
        if tot < r.peso_enlace: add(eid, 3, r.peso_enlace - tot)
    for eid, vals in list(src.items()):                          # … y pasaje propio
        if eid.startswith("chunk:"): continue
        ent = r.entradas.get(eid)
        if ent and ent.get("pasaje_propio"):
            n = f"chunk:{ent['pasaje_propio']}"
            if n in r.idx: add(n, 3, r.peso_enlace * sum(vals))
    tot = {n: sum(v) for n, v in src.items()}; Z = sum(tot.values()); pf = [sum(v[k] for v in src.values()) for k in range(4)]
    print(f"  masa sin normalizar ‖v‖₁ = {Z:.2f}: (a) hechos aprobados {pf[0]:.2f} · (b) presa densa {pf[1]:.2f} ({pf[1] / Z * 100:.0f} % de la masa, repartida en {sum(1 for v in src.values() if v[1] > 0)} pasajes) · (c) anclas {pf[2]:.2f} · (d) enlace + pasaje propio {pf[3]:.2f}")
    print(pd.DataFrame([(str(n), *[round(x, 3) for x in v], round(tot[n], 3), round(tot[n] / Z, 4), marca(n[6:], oro) if n.startswith("chunk:") else "")
                        for n, v in sorted(src.items(), key=lambda x: -tot[x[0]])[:top]],
                       columns=["nodo", "(a)", "(b)", "(c)", "(d)", "total", "v norm.", ""]).to_string(index=False))
    pers = r._semillas(Q, [qv], stilde, apr)
    print(f"  comprobación contra _semillas() del sistema: diferencia máxima {max(abs(pers.get(n, 0) - tot.get(n, 0)) for n in set(pers) | set(tot)):.1e}")
    return pers, stilde

def etapa_paseo(pers, stilde, oro, k=10, fijo=False):
    p = r._ppr(pers, np.full(len(stilde), 1.0) if fijo else stilde)
    orden = [i for i in np.argsort(-p) if r.nodos[i].startswith("chunk:")][:60]; top = [r.nodos[i][6:] for i in orden]
    print(pd.DataFrame([(j + 1, t, round(float(p[i]), 4), marca(t, oro)) for j, (i, t) in enumerate(zip(orden[:k], top[:k]))],
                       columns=["puesto", "pasaje", "p*", ""]).to_string(index=False))
    return top

def etapa_salto(huecos, apr, enlaces):
    buscables = [h for h in huecos if h["anchor"] or "[" not in h["query"]]
    print(f"  huecos buscables ahora: {buscables} (los que dependen de un [X] esperan a la re-planificación)")
    if not buscables:
        print("  ningún hueco tiene ancla conocida: no hay salto (el sistema hace lo mismo: buscar() sale del bucle)")
        return []
    n0 = r.n_llamadas
    extra = [j for j in r._salto(buscables, apr, enlaces) if j not in apr]
    print(f"  mini-filtro del salto ({'en caché' if r.n_llamadas == n0 else 'NUEVA'}): {len(extra)} hechos nuevos aprobados")
    for j in extra: print(f"     + «{desc(j)[:90]}»  [de «{r.aser_ids[j].rsplit('::', 1)[0]}»]")
    return extra

def etapa_conjunto(Q, plan, top, oro):
    n0 = r.n_llamadas
    final = r._conjunto(Q, "; ".join(plan) if plan else Q, top)
    print(f"  conjunto final ({'en caché' if r.n_llamadas == n0 else 'NUEVA'}): top-5 = " + " | ".join(f"{t}{' ★' if t in oro else ''}" for t in final[:5]))
    return final

def metricas(top, oro, ks=(2, 5, 10)):
    if not oro:
        print("  (sin oro conocido: no hay métricas)"); return
    print("  " + "   ".join(f"R@{k} = {len(oro & set(top[:k])) / len(oro):.2f}, FC@{k} = {int(len(oro & set(top[:k])) == len(oro))}" for k in ks))

def trazar(q, puentes=(), top_siembra=8):
    Q = q["question"]; oro = titulos_oro(q)
    print(f"{Q}\n  pasajes oro: {sorted(oro)}\n")
    print("1) ENLAZADOR"); L = etapa_enlazador(Q)
    print("\n2) AFINIDAD"); qv, sig = etapa_afinidad(Q)
    print("\n3) CANDIDATOS K(q)"); K = etapa_candidatos(L, sig, puentes)
    print("\n4) ANALISTA"); apr, huecos, enlaces, plan = etapa_analista(Q, sig, L)
    print("\n5) SIEMBRA"); pers, stilde = etapa_siembra(Q, qv, sig, apr, oro, top=top_siembra)
    print("\n6) PASEO"); top = etapa_paseo(pers, stilde, oro); metricas(top, oro)
    ronda = 0
    while huecos and ronda < r.max_rondas:
        ronda += 1; print(f"\n7) SALTO DIRIGIDO, ronda {ronda}")
        extra = etapa_salto(huecos, apr, enlaces)
        if not extra: break
        apr = apr + extra
        print("   re-siembra y nuevo paseo:"); pers, stilde = etapa_siembra(Q, qv, sig, apr, oro, top=top_siembra); top = etapa_paseo(pers, stilde, oro); metricas(top, oro)
        pendientes = [h for h in huecos if not (h["anchor"] or "[" not in h["query"])]
        if not pendientes or ronda >= r.max_rondas: break
        print("   quedan huecos con comodín → el analista re-planifica viendo los hechos descubiertos:")
        sel2, huecos, enlaces, plan2 = etapa_analista(Q, sig, enlaces, extra=apr); plan = plan2 or plan
        nuevos = [j for j in sel2 if j not in apr]
        if nuevos:
            apr = apr + nuevos; pers, stilde = etapa_siembra(Q, qv, sig, apr, oro, top=top_siembra); top = etapa_paseo(pers, stilde, oro); metricas(top, oro)
    print("\n8) CONJUNTO FINAL"); final = etapa_conjunto(Q, plan, top, oro); metricas(final, oro)
    return final

print("funciones de traza definidas")

funciones de traza definidas


## 5. Dos preguntas reales del banco

### 5.1 Puente: *«What 2003 Christmas-themed romantic comedy did Gregor Fisher have a role in?»*

La pregunta nombra una sola entidad (Gregor Fisher); la película es el hueco. Obsérvese cómo el enlazador ancla la
entidad, cómo el analista escribe el plan con el comodín y cómo el hueco se rellena —por el propio Top-40 o por el salto—.

In [12]:
final_puente = trazar(q_puente, puentes=("Love Actually is a 2003",))

What 2003 Christmas-themed romantic comedy did Gregor Fisher have a role in?
  pasajes oro: ['Gregor Fisher', 'Love Actually']

1) ENLAZADOR
  tramos por mayúsculas: ['2003 Christmas-themed', 'Gregor Fisher']
    «2003 Christmas-themed» → plegado «2003 christmas themed» → escalón 1: nada
       escalón 2, mejores por Dice de trigramas: [('the christmas tree', 0.62), ('christmas', 0.6), ('this christmas', 0.57)]
    «Gregor Fisher» → plegado «gregor fisher» → escalón 1: ['can:gregor fisher']
  L(q) = ['can:gregor fisher']

2) AFINIDAD
  σ_a = <v_0, e(τ_a)> sobre 87629 hechos: máx 0.700 · media 0.109
    σ                                                             hecho τ_a                               entidades
0.700                   Gregor Fisher had a role in the film Love Actually.  [can:gregor fisher, can:love actually]
0.614                      Gregor Fisher starred in the 2006 movie Missing.  [can:gregor fisher, can:missing#16876]
0.608                                         

### 5.2 Comparación: *«Which rock band chose its name by drawing it out of a hat, Switchfoot or Midnight Oil?»*

Dos entidades nombradas y ningún puente: la cadena la resuelven el enlazador (dos anclas exactas) y el pasaje propio;
el analista selecciona los hechos de las dos ramas. En la v4 el conjunto final también corre aquí (antes se saltaba
en comparación), lo que en el sondeo de HotpotQA dio +2 puntos de FC@5 en este tipo.

In [13]:
final_comp = trazar(q_comp)

Which rock band chose its name by drawing it out of a hat, Switchfoot or Midnight Oil?
  pasajes oro: ['Midnight Oil', 'Switchfoot']

1) ENLAZADOR
  tramos por mayúsculas: ['Switchfoot', 'Midnight Oil']
    «Switchfoot» → plegado «switchfoot» → escalón 1: ['can:switchfoot']
    «Midnight Oil» → plegado «midnight oil» → escalón 1: ['can:midnight oil#29153', 'can:midnight oil#29158']
  L(q) = ['can:switchfoot', 'can:midnight oil#29153']

2) AFINIDAD
  σ_a = <v_0, e(τ_a)> sobre 87629 hechos: máx 0.675 · media 0.093
    σ                                                                         hecho τ_a                                      entidades
0.675 Farm changed its name to Midnight Oil by drawing the name out of a hat in Sydney. [can:farm, can:sydney, can:midnight oil#29158]
0.603                                 Midnight Oil is a prominent Australian rock band.                       [can:midnight oil#29158]
0.602                                          Midnight Oil is an Australian 

In [14]:
# comprobación: la traza reproduce exactamente lo que devuelve el sistema (mismas llamadas → misma caché)
for q, fin in ((q_puente, final_puente), (q_comp, final_comp)):
    sistema = r.buscar(q["question"], 20)
    print(f"{'coincide' if sistema[:5] == fin[:5] else 'DIFIERE'}: top-5 de buscar() = {sistema[:5]}")

coincide: top-5 de buscar() = ['Gregor Fisher', 'Love Actually', 'The Baldy Man', 'Rab C. Nesbitt', 'Andrew Lincoln']
coincide: top-5 de buscar() = ['Midnight Oil', 'Switchfoot', 'Rob Hirst', 'Read About It', 'Power and the Passion (song)']


## 6. Contrafactuales: qué cambia en la misma pregunta

Tres conmutadores del sistema sobre la pregunta de puente: **sin salto dirigido** (`usar_salto=False`), **sin conjunto
final** (`usar_conjunto=False`) y **sin modulación** ($\mu\equiv 1$: la matriz de transición fija de HippoRAG).
En el sondeo de HotpotQA (200 preguntas) el salto aportó +2,0 de FC@5 (significativo) y el conjunto +4,0 de R@2.

In [15]:
Q = q_puente["question"]; oro = titulos_oro(q_puente)
def top5(cfg, **kw):
    for k, v in kw.items(): setattr(r, k, v)
    top = r.buscar(Q, 10)
    print(f"  {cfg:24s} top-5: " + " | ".join(f"{t}{'★' if t in oro else ''}" for t in top[:5]) + f"   FC@5={int(oro <= set(top[:5]))}")
top5("completo (v4)")
top5("sin salto dirigido", usar_salto=False); r.usar_salto = True
top5("sin conjunto final", usar_conjunto=False); r.usar_conjunto = True
# sin modulación: mismo vector de siembra, paseo con μ ≡ 1
L = r._enlazar_menciones(Q); r._enlaces_actuales = L; qv = r._qvecs(Q, "lite")[0]; sig = r.aser_emb @ qv
apr, huecos, enlaces, plan = r._analista(Q, sig, L); r._enlaces_actuales = enlaces
stilde = sig.copy(); stilde[apr] = 1.0; pers = r._semillas(Q, [qv], stilde, apr)
for etiqueta, fijo in (("paseo modulado (μ_a)", False), ("paseo FIJO (μ ≡ 1)", True)):
    p = r._ppr(pers, np.full(len(stilde), 1.0) if fijo else stilde)
    top = [r.nodos[i][6:] for i in np.argsort(-p) if r.nodos[i].startswith("chunk:")][:5]
    print(f"  {etiqueta:24s} top-5: " + " | ".join(f"{t}{'★' if t in oro else ''}" for t in top))

  completo (v4)            top-5: Gregor Fisher★ | Love Actually★ | The Baldy Man | Rab C. Nesbitt | Andrew Lincoln   FC@5=1
  sin salto dirigido       top-5: Gregor Fisher★ | Love Actually★ | The Baldy Man | Rab C. Nesbitt | Andrew Lincoln   FC@5=1
  sin conjunto final       top-5: Gregor Fisher★ | Love Actually★ | Andrew Lincoln | Letters to Santa (film) | The Baldy Man   FC@5=1
  paseo modulado (μ_a)     top-5: Gregor Fisher★ | Love Actually★ | Andrew Lincoln | Letters to Santa (film) | The Baldy Man
  paseo FIJO (μ ≡ 1)       top-5: Gregor Fisher★ | Love Actually★ | Andrew Lincoln | Letters to Santa (film) | The Baldy Man


## 7. Un fallo real del banco

De los 98 fallos de cadena completa a k=5 (96 de puente), en 90 falta un solo pasaje: el puente. Tomamos el primero
en el que el oro queda entre los puestos 6 y 10 y miramos por qué.

In [16]:
res = {f["_id"]: f for f in (json.loads(l) for l in open(W / "resultados_plan_v4_4omini_hotpot_benchmark.jsonl", encoding="utf-8"))}
fallos = [f for f in res.values() if f["full_chain@5"] < 1]
pos = collections.Counter()
for f in fallos:
    for t in titulos_oro(por_id[f["_id"]]) - set(f["top"][:5]):
        p = f["top"].index(t) + 1 if t in f["top"] else None
        pos["6-10" if p and p <= 10 else "11-20" if p else "fuera del top-20"] += 1
print(f"fallos FC@5: {len(fallos)} de {len(res)} · oros perdidos por posición: {dict(pos)}")
q_fallo = next(por_id[f["_id"]] for f in fallos if any(t in f["top"][5:10] for t in titulos_oro(por_id[f["_id"]])))
_ = trazar(q_fallo, top_siembra=6)

fallos FC@5: 98 de 1000 · oros perdidos por posición: {'fuera del top-20': 46, '11-20': 23, '6-10': 37}
What is the name of the former MGM Grand Garden Special Events Center where the Mayweather-Ortiz fight took place?
  pasajes oro: ['Canelo Álvarez vs. Alfonso Gómez', 'MGM Grand Garden Arena']

1) ENLAZADOR
  tramos por mayúsculas: ['MGM Grand Garden Special Events Center', 'Mayweather-Ortiz']
    «MGM Grand Garden Special Events Center» → plegado «mgm grand garden special events center» → escalón 1: nada
       escalón 2, mejores por Dice de trigramas: [('mgm grand garden arena', 0.54), ('mandalay bay events center', 0.43), ('grant medical center', 0.35)]
    «Mayweather-Ortiz» → plegado «mayweather ortiz» → escalón 1: nada
       escalón 2, mejores por Dice de trigramas: [('floyd mayweather jr', 0.57), ('leather', 0.43), ('victor ortiz', 0.43)]
  L(q) = ['can:mgm grand garden arena']

2) AFINIDAD
  σ_a = <v_0, e(τ_a)> sobre 87629 hechos: máx 0.785 · media 0.078
    σ               

## 8. El banco medido y las referencias

Agregados de la medición única (`plan_v4_4omini_hotpot_benchmark`) y del suelo BM25 sobre el mismo corpus, junto a
las cifras publicadas por CatRAG (Lau et al., Findings ACL 2026, tabla de su §4.4) **con los mismos modelos**.

In [17]:
def agg(nombre):
    a = json.load(open(W / f"agregados_{nombre}.json", encoding="utf-8"))
    fila = {"sistema": nombre, "R@2": a["recall@2"], "R@5": a["recall@5"], "FC@5": a["full_chain@5"], "FC@10": a["full_chain@10"]}
    for t, d in a["por_tipo"].items(): fila[f"FC@5 {t}"] = d["full_chain@5"]
    return fila
tabla = pd.DataFrame([agg("bm25_hotpot_benchmark"), agg("plan_v4_4omini_hotpot_benchmark")]).set_index("sistema") * 100
print(tabla.round(1).to_string())
print("\nreferencias publicadas (CatRAG, mismos modelos gpt-4o-mini + te3-small; FCR = FC@5):")
print(pd.DataFrame([("BM25 (su implementación)", 64.8, 38.3), ("denso te3-small", 81.3, 64.9), ("RAPTOR", 79.5, 61.4),
                    ("HippoRAG 2", 87.1, 75.5), ("CatRAG", 89.5, 80.4), ("Plan A v4 (este trabajo)", 94.7, 90.2)],
                   columns=["sistema", "R@5", "FC@5"]).to_string(index=False))
par = json.load(open(W / "pareado_plan_v4_4omini_hotpot_benchmark__vs__bm25_hotpot_benchmark.json", encoding="utf-8"))
m = par["metricas"]["full_chain@5"]
print(f"\npareado vs BM25 (bootstrap, 1 000 preguntas): ΔFC@5 = {100*m['dif']:+.1f} [{100*m['ic'][0]:+.1f}, {100*m['ic'][1]:+.1f}] {m['sig']} · gana {m['gana']} / pierde {m['pierde']}")

                                  R@2   R@5  FC@5  FC@10  FC@5 bridge  FC@5 comparison
sistema                                                                               
bm25_hotpot_benchmark            56.2  72.8  49.5   73.3         45.4             67.2
plan_v4_4omini_hotpot_benchmark  77.1  94.7  90.2   93.6         88.2             98.9

referencias publicadas (CatRAG, mismos modelos gpt-4o-mini + te3-small; FCR = FC@5):
                 sistema  R@5  FC@5
BM25 (su implementación) 64.8  38.3
         denso te3-small 81.3  64.9
                  RAPTOR 79.5  61.4
              HippoRAG 2 87.1  75.5
                  CatRAG 89.5  80.4
Plan A v4 (este trabajo) 94.7  90.2

pareado vs BM25 (bootstrap, 1 000 preguntas): ΔFC@5 = +40.7 [+37.1, +44.1] SIG · gana 432 / pierde 25


### Pregunta libre

Cambia `Q_libre` (en inglés, sobre el corpus de HotpotQA). Coste: una incrustación y dos o tres llamadas a `gpt-4o-mini`
(céntimos). Sin oro conocido, las columnas de marca quedan vacías.

In [18]:
Q_libre = "Which film written and directed by Richard Curtis features Gregor Fisher in its ensemble cast?"
_ = trazar({"question": Q_libre, "supporting_facts": []}, top_siembra=6)

Which film written and directed by Richard Curtis features Gregor Fisher in its ensemble cast?
  pasajes oro: []

1) ENLAZADOR
  tramos por mayúsculas: ['Richard Curtis', 'Gregor Fisher']
    «Richard Curtis» → plegado «richard curtis» → escalón 1: ['can:richard curtis']
    «Gregor Fisher» → plegado «gregor fisher» → escalón 1: ['can:gregor fisher']
  L(q) = ['can:richard curtis', 'can:gregor fisher']

2) AFINIDAD
  σ_a = <v_0, e(τ_a)> sobre 87629 hechos: máx 0.664 · media 0.099
    σ                                                                             hecho τ_a                                             entidades
0.664                                   Gregor Fisher had a role in the film Love Actually.                [can:gregor fisher, can:love actually]
0.613                             Love Actually was written and directed by Richard Curtis.               [can:love actually, can:richard curtis]
0.605 Four Weddings and a Funeral was the first of several films by screenwri

### Lote opcional con el harness

`evaluar(buscar, preguntas)` calcula R@k y FC@k por pregunta y agregados con intervalos bootstrap. Desactivado por defecto;
ponlo a `True` para medir las 20 primeras preguntas del sondeo de HotpotQA (≈0,02 USD, un par de minutos; carga otro índice).

In [19]:
EJECUTAR_LOTE = False
if EJECUTAR_LOTE:
    from asistente_vih.eval.wiki2 import evaluar, imprimir
    rs = Wiki2PlanRAG.v4(split="hotpot_sondeo", modelo="gpt-4o-mini")
    preg_s, _ = cargar_split("hotpot_sondeo")
    imprimir(evaluar(rs.buscar, preg_s[:20], nombre="cuaderno_hotpot_sondeo_20", guardar=False))
else:
    print("lote desactivado (EJECUTAR_LOTE = False)")

lote desactivado (EJECUTAR_LOTE = False)


## Correspondencia con el documento y con el código

- Bancos y protocolo: §sec:protocolo y §sec:generalizacion (tabla de referencias `tab:otros-datasets`, hipótesis H1–H3).
- Índice: extracción §sec:openie · memoria canónica §sec:diccionario y §sec:memoria (E1–E9) · grafo por capas §sec:grafo · representación dual §sec:dual.
- Consulta: enlazador §sec:linker (L1–L4) · afinidad §sec:afinidad · K(q) y filtro §sec:filtro · siembra §sec:siembra · paseo y forma cerrada §sec:ppr · Plan A §sec:evoluciones · **v4 y HotpotQA §sec:v4**.
- Código: `eval/wiki2.py` (bancos, splits, métricas) · `ingest/wiki2_openie.py`, `wiki2_entidades.py`, `wiki2_grafo.py` (índice) · `retrieval/wiki2_catrag.py` (enlazador, siembra, paseo) · `retrieval/wiki2_plan.py` (analista, salto, conjunto) · `eval/wiki2_correr.py` (mediciones) · `eval/wiki2_auditoria.py` (auditoría con QIDs, solo 2Wiki).
- Trazabilidad de la sesión: `artifacts/wiki2/BITACORA_2026-08-30.md`.